In [287]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)

## 1) Generate synthetic population

In [288]:
def generate_synthetic_population():
    N = 10_000 # population size or number of rows
    
    # talako lists lai dataframe create garna use gariyeko xa
    regions = ["Bagmati", "Gandaki", "Koshi", "Lumbini", "Madhesh", "Karnali", "Sudurpashchim"]
    urban_rural = ["Urban", "Rural"]
    sexes = ["Male", "Female"]
    education_levels = ["None", "Primary", "Secondary", "Higher"]
    occupations = ["Agriculture", "Construction", "Manufacturing", "Trade", "Services", "Finance", "Education", "Health"]
    employment_statuses = ["Employed", "Self-employed", "Unemployed", "Student"]

    # mathi ko lists bata random selection garera dataframe (10_000 X 8) banaiyeko xa
    pop = pd.DataFrame({
        "person_id": np.arange(1, N+1), #unique id for each person
        #randomly assign values on the basis of probability for reagion,urban_rural,sex,education,employment_status 
        "region": np.random.choice(regions, size=N, p=[0.35, 0.14, 0.12, 0.12, 0.09, 0.10, 0.08]),
        "urban_rural": np.random.choice(urban_rural, size=N, p=[0.65, 0.35]),
        #randomly assign numbers from 1 to 10 
        "household_size": np.random.randint(1, 10, size=N),
        #Ages from normal distribution (mean 35, sd 15), clipped to 5-90, rounded, and converted to int
        "age": np.random.normal(35, 15, size=N).clip(5, 90).round().astype(int),
        "sex": np.random.choice(sexes, size=N, p=[0.51, 0.49]),
        "education": np.random.choice(education_levels, size=N, p=[0.10, 0.30, 0.40, 0.20]),
        "employment_status": np.random.choice(employment_statuses, size=N, p=[0.40, 0.25, 0.15, 0.20]),
    })

    # ────────────────────────────────────────────────
    # Occupation: only for people who are working
    # ────────────────────────────────────────────────
    pop["occupation"] = "Not applicable" #new column filled with "Not applicable" add hunxa

    #Boolean mask (True/False Series) where status is Employed or Self-employed
    working_mask = pop["employment_status"].isin(["Employed", "Self-employed"])

    #Counts True values (number of working people)
    n_working = working_mask.sum()

    #working rows lai matra occupation assign garinxa, with probabilities
    pop.loc[working_mask, "occupation"] = np.random.choice(
        occupations,
        size=n_working,
        p=[0.30, 0.10, 0.10, 0.12, 0.15, 0.08, 0.08, 0.07]
    )

    # ────────────────────────────────────────────────
    # Wage logic (wage = occ_base * edu_mult * urban_rural)
    # ────────────────────────────────────────────────

    #base wages per occupation ko list 
    occ_base = {
        "Agriculture": 38000,
        "Construction": 55000,
        "Manufacturing": 62000,
        "Trade": 58000,
        "Services": 72000,
        "Finance": 80000,
        "Education": 50000,
        "Health": 89000,
        "Not applicable": 0
    }

    # Education multipliers (higher education le wage effect garxa)
    edu_mult = {"None": 0.70, "Primary": 0.90, "Secondary": 1.15, "Higher": 1.60}

    # Adds wage column by mapping occupation to base value, converts to float.
    pop["wage"] = pop["occupation"].map(occ_base).astype(float)

    #Multiplies by 1.3 if Urban, else 1.0.
    pop["wage"] *= np.where(pop["urban_rural"] == "Urban", 1.30, 1.00)

    # add variation only to workers.
    pop.loc[working_mask, "wage"] += np.random.normal(0, 18000, size=working_mask.sum())

    # Students and Unemployed wage 

    #Mask for Unemployed
    unemploy_mask = pop["employment_status"].isin(["Unemployed"])
    #Overwrites unemployed wages with small random values for pocket money, mini task, passive income
    pop.loc[pop["employment_status"] == "Unemployed",   "wage"] = np.random.normal(0, 10000, size=unemploy_mask.sum())

    #Mask for Student
    student_mask = pop["employment_status"].isin(["Student"])
    #Same as unemployed but smaller variation
    pop.loc[pop["employment_status"] == "Student",  "wage"] = np.random.normal(0, 5000, size=student_mask.sum())

    # Multiplies wage by education factor for all rows
    pop["wage"] *= pop["education"].map(edu_mult)
    
    # Final cleaning, Clips negatives to 0, rounds to integer, converts to int type.
    pop["wage"] = pop["wage"].clip(lower=0).round(0).astype(int)
    
    return pop

pop =  generate_synthetic_population()
print(pop.head())

   person_id   region urban_rural  household_size  age     sex  education  \
0          1  Gandaki       Rural               6   38    Male  Secondary   
1          2  Gandaki       Rural               6   32  Female    Primary   
2          3  Madhesh       Urban               3   29    Male    Primary   
3          4  Karnali       Urban               6   67    Male  Secondary   
4          5  Karnali       Urban               1   17  Female    Primary   

  employment_status    occupation   wage  
0     Self-employed  Construction  61204  
1     Self-employed         Trade  32180  
2     Self-employed   Agriculture  82088  
3          Employed   Agriculture  90968  
4          Employed         Trade  53317  


### Population summary function

In [289]:
def summarize(df, name):
    return {
        "name": name, #name passed
        "n": len(df), #Number of rows
        "mean_wage": df["wage"].mean().round(2), #Average wage, rounded
        "median_wage": df["wage"].median().round(2), #Median wage, rounded
        "mean_age": df["age"].mean().round(), #Average age, rounded
        "mean_hh_size": df["household_size"].mean().round(), #Average household size, rounded
        "sex_dist": df["sex"].value_counts(normalize=True).round(3).to_dict(), #Sex distribution as dictionary
        #Education distribution as dictionary
        "education_dist": df["education"].value_counts(normalize=True).round(3).to_dict(), 
        #Occupation distribution as dictionary
        "occupation_dist": df["occupation"].value_counts(normalize=True).round(3).to_dict(), 
        "region_dist": df["region"].value_counts(normalize=True).round(3).to_dict(), #Region distribution as dictionary
        #Urban/rural distribution as dictionary
        "urban_rural_dist": df["urban_rural"].value_counts(normalize=True).round(3).to_dict(), 
        #Employment status distribution as dictionary
        "emp_status_dist": df["employment_status"].value_counts(normalize=True).round(3).to_dict(), 
    }

pop_summary = summarize(pop, "Population")
pop_summary

{'name': 'Population',
 'n': 10000,
 'mean_wage': 51656.67,
 'median_wage': 49119.5,
 'mean_age': 35.0,
 'mean_hh_size': 5.0,
 'sex_dist': {'Male': 0.501, 'Female': 0.499},
 'education_dist': {'Secondary': 0.393,
  'Primary': 0.303,
  'Higher': 0.204,
  'None': 0.099},
 'occupation_dist': {'Not applicable': 0.341,
  'Agriculture': 0.204,
  'Services': 0.097,
  'Trade': 0.079,
  'Manufacturing': 0.067,
  'Construction': 0.066,
  'Finance': 0.051,
  'Education': 0.049,
  'Health': 0.047},
 'region_dist': {'Bagmati': 0.346,
  'Gandaki': 0.143,
  'Lumbini': 0.122,
  'Koshi': 0.119,
  'Karnali': 0.103,
  'Madhesh': 0.089,
  'Sudurpashchim': 0.078},
 'urban_rural_dist': {'Urban': 0.654, 'Rural': 0.346},
 'emp_status_dist': {'Employed': 0.404,
  'Self-employed': 0.255,
  'Student': 0.198,
  'Unemployed': 0.143}}

## 2) Sampling functions

In [ ]:
#Randomly selects n rows using Pandas .sample, resets index.
def simple_random_sampling(df, n, seed=42):
    # sabai row have equal chances
    return df.sample(n=n, random_state=seed).reset_index(drop=True)

#Shuffles df, calculates interval k, starts from random point, takes every k-th row.
def systematic_sampling(df, n, seed=42):
    d = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    k = max(len(d) // n, 1)
    start = seed % k
    return d.iloc[start::k].head(n).reset_index(drop=True)

#Calculates proportions by strata (e.g., education), allocates sample sizes proportionally, samples within each stratum.
def stratified_sampling(df, strata_col, n, seed=42):
    rng = np.random.default_rng(seed)
    props = df[strata_col].value_counts(normalize=True)
    alloc = (props * n).round().astype(int)
    diff = n - alloc.sum()
    if diff != 0:
        order = props.sort_values(ascending=False).index
        i = 0
        while diff != 0:
            s = order[i % len(order)]
            if diff > 0:
                alloc[s] += 1
                diff -= 1
            elif alloc[s] > 0:
                alloc[s] -= 1
                diff += 1
            i += 1

    parts = []
    for s, k in alloc.items():
        group = df[df[strata_col] == s]
        k = min(k, len(group))
        if k > 0:
            idx = rng.choice(group.index.to_numpy(), size=k, replace=False)
            parts.append(df.loc[idx])
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

#Randomly selects n_clusters (e.g., regions), takes all rows from those clusters.
def cluster_sampling(df, cluster_col, n_clusters, seed=42):
    rng = np.random.default_rng(seed)
    clusters = df[cluster_col].unique()
    chosen = rng.choice(clusters, size=min(n_clusters, len(clusters)), replace=False)
    return df[df[cluster_col].isin(chosen)].reset_index(drop=True)

#Takes first n rows by ID (easiest to "access").
def convenience_sampling(df, n):
    return df.sort_values("person_id").head(n).reset_index(drop=True)

#Samples fixed numbers from quotas (e.g., 255 Males).
def quota_sampling(df, quotas, seed=42):
    # Example: quotas = {"sex": {"Male":255, "Female":245}}
    rng = np.random.default_rng(seed)
    pieces = []
    for col, qmap in quotas.items():
        for val, k in qmap.items():
            group = df[df[col] == val]
            k = min(k, len(group))
            if k > 0:
                idx = rng.choice(group.index.to_numpy(), size=k, replace=False)
                pieces.append(df.loc[idx])
    return pd.concat(pieces).drop_duplicates("person_id").sample(frac=1, random_state=seed).reset_index(drop=True)

#Simulates network: starts from seed IDs, expands via random "adjacency" until n reached.
def snowball_sampling(df, seed_ids, n, seed=42):
    rng = np.random.default_rng(seed)
    ids = df["person_id"].to_numpy()
    adjacency = {pid: rng.choice(ids, size=int(rng.integers(3,9)), replace=False).tolist() for pid in ids}

    selected = set()
    queue = list(seed_ids)

    while queue and len(selected) < n:
        pid = queue.pop(0)
        if pid in selected:
            continue
        selected.add(pid)
        for nb in adjacency.get(pid, []):
            if nb not in selected:
                queue.append(nb)

    return df[df["person_id"].isin(selected)].reset_index(drop=True)

## 3) Draw samples

In [291]:
def draw_samples(pop):
    samples = {}
    samples["Simple Random"] = simple_random_sampling(pop, 500)
    samples["Systematic"] = systematic_sampling(pop, 500)
    samples["Stratified (emp_status)"] = stratified_sampling(pop, "employment_status", 500)
    samples["Cluster (2 regions)"] = cluster_sampling(pop, "region", 2)
    samples["Convenience"] = convenience_sampling(pop, 500)
    samples["Quota (sex)"] = quota_sampling(pop, {"sex": {"Male": 255, "Female": 245}}, seed=42)
    samples["Snowball"] = snowball_sampling(pop, seed_ids=[1,2,3,4,5], n=500)
    return samples

samples = draw_samples(pop)

## 4) Compare summaries

In [292]:
summaries = [pop_summary] + [summarize(df, name) for name, df in samples.items()]
summary_df = pd.DataFrame(summaries).set_index("name")
summary_df.round(2)

,n,mean_wage,median_wage,mean_age,mean_hh_size,sex_dist,education_dist,occupation_dist,region_dist,urban_rural_dist,emp_status_dist
name,,,,,,,,,,,
Population,10000,51656.67,49119.5,35.0,5.0,"{'Male': 0.501, 'Female': 0.499}","{'Secondary': 0.393, 'Primary': 0.303, 'Higher...","{'Not applicable': 0.341, 'Agriculture': 0.204...","{'Bagmati': 0.346, 'Gandaki': 0.143, 'Lumbini'...","{'Urban': 0.654, 'Rural': 0.346}","{'Employed': 0.404, 'Self-employed': 0.255, 'S..."
Simple Random,500,52376.33,51855.0,34.0,5.0,"{'Female': 0.51, 'Male': 0.49}","{'Secondary': 0.394, 'Primary': 0.3, 'Higher':...","{'Not applicable': 0.334, 'Agriculture': 0.208...","{'Bagmati': 0.342, 'Gandaki': 0.134, 'Lumbini'...","{'Urban': 0.686, 'Rural': 0.314}","{'Employed': 0.43, 'Self-employed': 0.236, 'St..."
Systematic,500,54668.94,54621.0,35.0,5.0,"{'Male': 0.514, 'Female': 0.486}","{'Secondary': 0.382, 'Primary': 0.278, 'Higher...","{'Not applicable': 0.32, 'Agriculture': 0.198,...","{'Bagmati': 0.35, 'Gandaki': 0.144, 'Lumbini':...","{'Urban': 0.658, 'Rural': 0.342}","{'Employed': 0.396, 'Self-employed': 0.284, 'S..."
Stratified (emp_status),500,51537.61,45217.5,35.0,5.0,"{'Male': 0.502, 'Female': 0.498}","{'Secondary': 0.38, 'Primary': 0.294, 'Higher'...","{'Not applicable': 0.342, 'Agriculture': 0.216...","{'Bagmati': 0.318, 'Gandaki': 0.142, 'Koshi': ...","{'Urban': 0.646, 'Rural': 0.354}","{'Employed': 0.402, 'Self-employed': 0.256, 'S..."
Cluster (2 regions),2615,51284.26,49569.0,36.0,5.0,"{'Male': 0.509, 'Female': 0.491}","{'Secondary': 0.392, 'Primary': 0.308, 'Higher...","{'Not applicable': 0.346, 'Agriculture': 0.199...","{'Gandaki': 0.546, 'Koshi': 0.454}","{'Urban': 0.635, 'Rural': 0.365}","{'Employed': 0.394, 'Self-employed': 0.26, 'St..."
Convenience,500,55806.06,53721.0,35.0,5.0,"{'Female': 0.51, 'Male': 0.49}","{'Secondary': 0.418, 'Primary': 0.298, 'Higher...","{'Not applicable': 0.3, 'Agriculture': 0.178, ...","{'Bagmati': 0.374, 'Gandaki': 0.134, 'Lumbini'...","{'Urban': 0.636, 'Rural': 0.364}","{'Employed': 0.424, 'Self-employed': 0.276, 'S..."
Quota (sex),500,53024.48,52196.0,35.0,5.0,"{'Male': 0.51, 'Female': 0.49}","{'Secondary': 0.372, 'Primary': 0.276, 'Higher...","{'Not applicable': 0.318, 'Agriculture': 0.214...","{'Bagmati': 0.342, 'Gandaki': 0.14, 'Karnali':...","{'Urban': 0.678, 'Rural': 0.322}","{'Employed': 0.376, 'Self-employed': 0.306, 'S..."
Snowball,500,50220.66,43465.5,36.0,5.0,"{'Female': 0.53, 'Male': 0.47}","{'Secondary': 0.378, 'Primary': 0.296, 'Higher...","{'Not applicable': 0.386, 'Agriculture': 0.16,...","{'Bagmati': 0.322, 'Gandaki': 0.152, 'Lumbini'...","{'Urban': 0.678, 'Rural': 0.322}","{'Employed': 0.386, 'Self-employed': 0.228, 'S..."


## 5) Mean wage error comparison

In [293]:
pop_mean = pop["wage"].mean()

errors = []
for name, df in samples.items():
    err = df["wage"].mean() - pop_mean
    errors.append({"sample": name, "n": len(df), "mean_wage_error": round(err, 2), "abs_error": round(abs(err), 2)})

pd.DataFrame(errors).sort_values("abs_error").reset_index(drop = True)

,sample,n,mean_wage_error,abs_error
0,Stratified (emp_status),500,-119.06,119.06
1,Cluster (2 regions),2615,-372.40,372.40
2,Simple Random,500,719.66,719.66
3,Quota (sex),500,1367.81,1367.81
4,Snowball,500,-1436.00,1436.00
5,Systematic,500,3012.28,3012.28
6,Convenience,500,4149.39,4149.39


In [294]:
num_runs = 20
all_errors = []
for run in range(num_runs):
    pop = generate_synthetic_population()
    samples = draw_samples(pop)
    pop_mean = pop["wage"].mean()
    for name, df in samples.items():
        err = df["wage"].mean() - pop_mean
        all_errors.append({"run": run, "sample": name, "n": len(df), "mean_wage_error": round(err, 2), "abs_error": round(abs(err), 2)})

# ── Convert to DataFrame and summarize ──
errors_df = pd.DataFrame(all_errors)

# Summary table: average absolute error per method
summary = errors_df.groupby("sample").agg(
    avg_abs_error=("abs_error", "mean"),
    std_abs_error=("abs_error", "std"),
    avg_error=("mean_wage_error", "mean"),
    count=("run", "count")
).round(2).sort_values("avg_abs_error")

display(summary)

,avg_abs_error,std_abs_error,avg_error,count
sample,,,,
Cluster (2 regions),489.94,392.91,197.22,20
Stratified (emp_status),1058.98,905.31,-41.11,20
Snowball,1194.45,1303.10,-76.76,20
Convenience,1448.37,1110.08,-155.72,20
Quota (sex),1483.68,1092.22,14.30,20
Systematic,1535.06,1191.57,766.24,20
Simple Random,1572.57,985.45,267.05,20


## Notes
Stratified (education) and Cluster sampling showed the lowest bias in mean wage estimation. Non-probability methods had significantly higher error.
